# 第十一课｜为什么不能每次 spike 都扫描所有突触？

上一课 queue 里已经有 `source_id`。现在系统要回答：这个 source spike 应该送给哪些 target？今天只解决：
> **怎样只访问真实存在的连接，而不是检查所有可能的 neuron pair？**

主要新概念：**稀疏图表示（sparse graph representation）**。


## 1. 概念账本

**已经知道：** spike event 带 source_id；queue 能保存事件顺序。

**今天学习：** sparse graph、adjacency list，以及 **Compressed Sparse Row (CSR，压缩稀疏行)** 的核心思想。

**只预告：** 下一课才把 lookup、weighted event、target update 串成完整旅程；DDR 仍然很远。


## 2. 把神经网络先看成 graph

这里的 **graph（图）** 只有两个基本东西：

- neuron 是 node；
- synapse 是有方向的 edge。

如果 4 个 neuron 只有 4 条 synapse，连接其实很稀疏。用 4×4 dense matrix 表示时会有 16 个位置，但绝大多数是“没有 edge”。


## 3. adjacency list：只写真实存在的 edge

**邻接表（adjacency list）**：为每个 source 直接列出它真正连接到的 target。

例如：

- source 0 → `(1,+2)`, `(3,-1)`
- source 1 → none
- source 2 → `(1,+3)`
- source 3 → `(2,+1)`

一次 source 0 spike 只需要访问两条记录，不需要把 0→0、0→1、0→2、0→3 全部重新判断。


## 4. CSR 的核心：连续 records + source 索引

**压缩稀疏行（Compressed Sparse Row, CSR）** 常把同一 row 的非零项连续存储，再用 index 告诉我们每一 row 从哪里开始。

在本项目里，source neuron 可以类比 row。MDD 当前定义的是 `source_index = (start_offset, fanout_count)` 加连续的 `synapse_records`，与 CSR 的核心思想一致。

```mermaid
flowchart LR
 S["source_id"] --> IDX["source_index: start,count"]
 IDX --> REC["contiguous synapse_records"]
 REC --> O["target, weight ..."]
```


## 5. Run：把四条 edge 压成连续记录

先预测 source 1 的 `count`，再运行。


In [ ]:
edges = [
    (0, 1, 2),
    (0, 3, -1),
    (2, 1, 3),
    (3, 2, 1),
]
num_sources = 4

records = []
source_index = []
for source in range(num_sources):
    start = len(records)
    for src, target, weight in edges:
        if src == source:
            records.append((target, weight))
    source_index.append((start, len(records) - start))

print('source_index =', source_index)
print('synapse_records =', records)
for source, (start, count) in enumerate(source_index):
    print(f'source {source}:', records[start:start+count])


## 6. Observe

结果应满足：

- source 0 指向 records 的前两项；
- source 1 的 count 为 0；
- source 2 / 3 各有一项；
- 所有真实 edge 只存一次。

这就是 T-009 “source index → exact synapse range” 想验证的核心性质。


## 7. Try It：比较扫描量

对 1000 个 neuron，如果某个 source 只有 3 个下游 synapse：

- dense scan 需要检查多少个可能 target？
- sparse lookup 在找到 range 后需要读多少条 synapse record？

这里只比较“需要看的记录数量”，还不讨论 memory latency、cache 或 DDR bandwidth。


## 8. 作业

完成 `exercises/lesson11_sparse_graph.py`：

- `build_source_index(...)` 把 edge 按 source 压成连续 records；
- `lookup_source(...)` 必须返回该 source 的精确 slice。

```bash
uv run pytest exercises/checks/check_lesson11.py -q
```


## 9. AI Task

给 AI 一组 5 个 source、少量 edge，让它生成 `source_index + records`。你必须要求它逐个 source 用 slice 反查，证明 index 没有 off-by-one。


## 10. Human Check

不用 AI，你应该能解释：dense matrix 为什么浪费扫描；adjacency list 为什么适合 sparse connectivity；`start_offset` 与 `fanout_count` 各表示什么；为什么 source 没有下游 edge 时 count=0 仍然是合法记录。


## 11. Engineering Handoff

本课对齐 `MOD-006 source_index_store` 与 `MOD-007 synapse_reader` 的数据组织直觉，以及 T-009。还没有冻结 binary image、record bit width、DDR layout 或正式 streaming interface。


## 12. 项目追踪 Project Trace

- Lesson: `LSN-011`
- Mapping: `RMD-008`
- Module context: `MOD-006 / MOD-007`
- Test context: `T-009`


## 13. Exit Ticket

给你一个 source_id 和 `(start,count)`，你能准确指出它应该读取 records 的哪一段，并解释为什么这比扫描所有可能突触更符合稀疏网络。
